<a href="https://colab.research.google.com/github/banerjee109/Infotact-Data-Team-2/blob/Soumodeep_EDA/Assignmenet_1_Roughbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy
!python generate_marketing_data.py

python3: can't open file '/content/generate_marketing_data.py': [Errno 2] No such file or directory


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

NUM_USERS = 2000
NUM_DAYS = 90
START_DATE = datetime(2024, 1, 1)

CHANNELS = ['Google_Search', 'Meta_Facebook', 'TikTok', 'LinkedIn', 'Email', 'Organic']
CAMPAIGNS = {
    'Google_Search': ['GSearch_Brand', 'GSearch_Generic', 'GSearch_Competitor'],
    'Meta_Facebook': ['FB_Retargeting', 'FB_Lookalike', 'FB_Awareness'],
    'TikTok': ['TikTok_Video1', 'TikTok_Promo'],
    'LinkedIn': ['LinkedIn_B2B', 'LinkedIn_Lead'],
    'Email': ['Email_Welcome', 'Email_Promo', 'Email_Retention'],
    'Organic': ['Organic_SEO', 'Organic_Direct']
}
UTM_SOURCES = {
    'Google_Search': 'google',
    'Meta_Facebook': 'facebook',
    'TikTok': 'tiktok',
    'LinkedIn': 'linkedin',
    'Email': 'email',
    'Organic': 'organic'
}
PAGES = ['/home', '/product', '/pricing', '/about', '/blog', '/checkout', '/signup']

# FILE 1 - Ad Spend
print("Generating Ad Spend Data...")
ad_spend_rows = []
for day_offset in range(NUM_DAYS):
    date = START_DATE + timedelta(days=day_offset)
    for channel in CHANNELS:
        for campaign in CAMPAIGNS[channel]:
            spend_range = {
                'Google_Search': (500, 3000),
                'Meta_Facebook': (300, 2000),
                'TikTok': (200, 1500),
                'LinkedIn': (400, 2500),
                'Email': (50, 300),
                'Organic': (0, 0)
            }
            low, high = spend_range[channel]
            spend = round(random.uniform(low, high), 2)
            clicks = int(spend / random.uniform(0.5, 3.0)) if spend > 0 else random.randint(50, 500)
            impressions = clicks * random.randint(10, 50)
            ad_spend_rows.append({
                'date': date.strftime('%Y-%m-%d'),
                'channel': channel,
                'campaign': campaign,
                'amount_spent_usd': spend,
                'clicks': clicks,
                'impressions': impressions,
                'cpc': round(spend / clicks, 3) if clicks > 0 else 0
            })
ad_spend_df = pd.DataFrame(ad_spend_rows)
ad_spend_df.to_csv('ad_spend_data.csv', index=False)
print(f"✅ ad_spend_data.csv created — {len(ad_spend_df)} rows")

# FILE 2 - Web Analytics
print("Generating Web Analytics Log...")
web_log_rows = []
user_ids = [f'USR_{str(i).zfill(5)}' for i in range(1, NUM_USERS + 1)]
for user_id in user_ids:
    num_touchpoints = random.randint(1, 5)
    journey_start = START_DATE + timedelta(days=random.randint(0, NUM_DAYS - 10))
    for tp in range(num_touchpoints):
        session_date = journey_start + timedelta(days=random.randint(0, 7))
        channel = random.choices(CHANNELS, weights=[30, 25, 15, 10, 10, 10])[0]
        campaign = random.choice(CAMPAIGNS[channel])
        utm_source = UTM_SOURCES[channel] if random.random() > 0.1 else None
        utm_medium = 'cpc' if channel in ['Google_Search', 'Meta_Facebook', 'TikTok', 'LinkedIn'] else 'email' if channel == 'Email' else 'organic'
        utm_campaign = campaign if random.random() > 0.05 else None
        web_log_rows.append({
            'session_id': f'SES_{random.randint(100000, 999999)}',
            'user_id': user_id,
            'timestamp': session_date.strftime('%Y-%m-%d') + f' {random.randint(0,23):02d}:{random.randint(0,59):02d}:00',
            'channel': channel,
            'utm_source': utm_source,
            'utm_medium': utm_medium,
            'utm_campaign': utm_campaign,
            'page_visited': random.choice(PAGES),
            'session_duration_sec': random.randint(30, 900),
            'device': random.choice(['mobile', 'desktop', 'tablet'])
        })
web_log_df = pd.DataFrame(web_log_rows)
web_log_df.to_csv('web_analytics_log.csv', index=False)
print(f"✅ web_analytics_log.csv created — {len(web_log_df)} rows")

# FILE 3 - CRM
print("Generating CRM Conversion Data...")
crm_rows = []
converting_users = random.sample(user_ids, int(NUM_USERS * 0.30))
for user_id in converting_users:
    user_sessions = web_log_df[web_log_df['user_id'] == user_id]
    if user_sessions.empty:
        continue
    last_session = user_sessions.sort_values('timestamp').iloc[-1]
    conversion_date = pd.to_datetime(last_session['timestamp']) + timedelta(days=random.randint(0, 2))
    revenue = round(random.uniform(20, 500), 2)
    last_channel = last_session['channel']
    crm_rows.append({
        'customer_id': user_id,
        'conversion_date': conversion_date.strftime('%Y-%m-%d'),
        'revenue_usd': revenue,
        'last_touch_channel': last_channel,
        'last_touch_campaign': last_session['utm_campaign'],
        'product_purchased': random.choice(['Basic Plan', 'Pro Plan', 'Enterprise Plan', 'One-Time Purchase']),
        'country': random.choice(['India', 'USA', 'UK', 'Germany', 'Australia', 'Canada'])
    })
crm_df = pd.DataFrame(crm_rows)
crm_df.to_csv('crm_conversion_data.csv', index=False)
print(f"✅ crm_conversion_data.csv created — {len(crm_df)} rows")

Generating Ad Spend Data...
✅ ad_spend_data.csv created — 1350 rows
Generating Web Analytics Log...
✅ web_analytics_log.csv created — 6034 rows
Generating CRM Conversion Data...
✅ crm_conversion_data.csv created — 600 rows


In [3]:
# FILE 3 - CRM Only
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# Ensure necessary data and variables are available if not defined in preceding cells.
# Assuming 'web_analytics_log.csv' has been generated by previous steps.
web_log_df = pd.read_csv('web_analytics_log.csv')
web_log_df['timestamp'] = pd.to_datetime(web_log_df['timestamp']) # Convert timestamp for sorting

user_ids = web_log_df['user_id'].unique().tolist()
NUM_USERS = len(user_ids)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print("Generating CRM Conversion Data...")
crm_rows = []
converting_users = random.sample(user_ids, int(NUM_USERS * 0.30))
for user_id in converting_users:
    user_sessions = web_log_df[web_log_df['user_id'] == user_id]
    if user_sessions.empty:
        continue
    last_session = user_sessions.sort_values('timestamp').iloc[-1]
    conversion_date = pd.to_datetime(last_session['timestamp']) + timedelta(days=random.randint(0, 2))
    revenue = round(random.uniform(20, 500), 2)
    last_channel = last_session['channel']
    crm_rows.append({
        'customer_id': user_id,
        'conversion_date': conversion_date.strftime('%Y-%m-%d'),
        'revenue_usd': revenue,
        'last_touch_channel': last_channel,
        'last_touch_campaign': last_session['utm_campaign'],
        'product_purchased': random.choice(['Basic Plan', 'Pro Plan', 'Enterprise Plan', 'One-Time Purchase']),
        'country': random.choice(['India', 'USA', 'UK', 'Germany', 'Australia', 'Canada'])
    })
crm_df = pd.DataFrame(crm_rows)
crm_df.to_csv('crm_conversion_data.csv', index=False)
print(f"✅ crm_conversion_data.csv created — {len(crm_df)} rows")
print("\n🎉 DONE! Teeno CSV files ban gayi!")
crm_df.head()
crm_df.tail()
crm_df.info()
crm_df.describe()
type(crm_df)
crm_df.shape
crm_df.columns
crm_df.head(25)
##|crm_df["conversion_date"].value_counts()
##crm_df[(crm_df["conversion_date"]=="2024-01-22")]["revenue_usd"]
crm_df.info()
crm_df["conversion_date"]=pd.to_datetime(crm_df["conversion_date"])
crm_df.info()
type(crm_df["conversion_date"])
crm_df["year"] = crm_df["conversion_date"].dt.year
crm_df["month"] = crm_df["conversion_date"].dt.month
crm_df["day"]=crm_df["conversion_date"].dt.day
crm_df[["year","month","day"]]
crm_df["quarter"]=crm_df["conversion_date"].dt.quarter
crm_df[["year","month","day","quarter"]]
crm_df.groupby(["year","quarter","month"])["revenue_usd"].sum().sort_values(ascending=False)


Generating CRM Conversion Data...
✅ crm_conversion_data.csv created — 600 rows

🎉 DONE! Teeno CSV files ban gayi!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          600 non-null    object 
 1   conversion_date      600 non-null    object 
 2   revenue_usd          600 non-null    float64
 3   last_touch_channel   600 non-null    object 
 4   last_touch_campaign  572 non-null    object 
 5   product_purchased    600 non-null    object 
 6   country              600 non-null    object 
dtypes: float64(1), object(6)
memory usage: 32.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          600 non-null    object 
 1   conversion_date     

year  quarter  month
2024  1        2        62489.29
               3        52172.45
               1        44004.11
Name: revenue_usd, dtype: float64

In [9]:
import pandas as pd
import numpy as np
ad_spend_df = pd.read_csv("ad_spend_data.csv")
ad_spend_df.head()
ad_spend_df.columns
len(ad_spend_df)
type(ad_spend_df)
ad_spend_df.shape
ad_spend_df.info()
ad_spend_df.describe()
ad_spend_df.head(10)
ad_spend_df["channel"].unique()
ad_spend_df["channel"].nunique()
ad_spend_df["campaign"].nunique()
ad_spend_df["amount_spent_usd"].idxmax()
ad_spend_df.iloc[1006]
ad_spend_df["amount_spent_usd"].idxmin()
ad_spend_df.iloc[[13,14,1006]]
ad_spend_df[(ad_spend_df["amount_spent_usd"]==0)]["campaign"].unique()
ad_spend_df.head()
ad_spend_df["impressions"].mean()
ad_spend_df[(ad_spend_df["impressions"]>=15000)]["campaign"].nunique()
##ad_spend_df.isnull().sum()
ad_spend_df.isna().sum()
ad_spend_df.head()
ad_spend_df.groupby("campaign")[["clicks","cpc"]].sum().sort_values(by="clicks",ascending=False)
ad_spend_df["channel"].unique()
ad_spend_df["channel"].nunique()
ad_spend_df.groupby("channel")["amount_spent_usd"].sum().sort_values(ascending=False)
ad_spend_df.groupby("channel")["amount_spent_usd"].mean().sort_values(ascending=False)
ad_spend_df["cost_per_click"] = ad_spend_df["amount_spent_usd"] / ad_spend_df["clicks"]
ad_spend_df["cost_per_click"].head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1350 entries, 0 to 1349
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              1350 non-null   object 
 1   channel           1350 non-null   object 
 2   campaign          1350 non-null   object 
 3   amount_spent_usd  1350 non-null   float64
 4   clicks            1350 non-null   int64  
 5   impressions       1350 non-null   int64  
 6   cpc               1350 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 74.0+ KB


,cost_per_click
0,0.562619
1,0.849031


In [15]:
import pandas as pd
import numpy as np
web_log_df=pd.read_csv("web_analytics_log.csv")
web_log_df.columns
web_log_df.head()
web_log_df["session_duration_sec"].describe()
web_log_df.head()
web_log_df["utm_medium"].nunique()
web_log_df.groupby(["utm_medium","device"])["session_duration_sec"].sum().sort_values(ascending=False)
web_log_df.head()
web_log_df["date"]=pd.to_datetime(web_log_df["timestamp"])
web_log_df.info()
web_log_df["date"].dt.day
web_log_df["date"].dt.day_name()
web_log_df["channel"].value_counts()
web_log_df.groupby("channel")["session_duration_sec"].mean().sort_values(ascending=False)
web_log_df["day"]=web_log_df["date"].dt.day_name()
web_log_df.groupby("day")["session_duration_sec"].sum().sort_values(ascending=False)
web_log_df["quarter"]=web_log_df["date"].dt.quarter
web_log_df.groupby("quarter")["session_duration_sec"].mean().sort_values(ascending=False)
web_log_df["utm_source"].nunique()
web_log_df["utm_source"].value_counts()
web_log_df.groupby("utm_source")["session_duration_sec"].mean().sort_values(ascending=False)
web_log_df.head()
web_log_df.groupby("day")["session_duration_sec"].sum().sort_values(ascending=False)
web_log_df.groupby("day")["session_duration_sec"].mean().sort_values(ascending=False)
web_log_df.isna().sum()
web_log_df.dropna(inplace=True)
##web_log_df.isna().sum()
len(web_log_df["utm_campaign"])
web_log_df["utm_source"].isna().sum()
web_log_df.groupby("user_id")["session_id"].count()
web_log_df.groupby(["utm_campaign","device"])["session_duration_sec"].sum().sort_values(ascending=False).reset_index()
web_log_df.groupby(["utm_campaign","device"])["session_duration_sec"].mean().sort_values(ascending=False).reset_index()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6034 entries, 0 to 6033
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   session_id            6034 non-null   object        
 1   user_id               6034 non-null   object        
 2   timestamp             6034 non-null   object        
 3   channel               6034 non-null   object        
 4   utm_source            5434 non-null   object        
 5   utm_medium            6034 non-null   object        
 6   utm_campaign          5735 non-null   object        
 7   page_visited          6034 non-null   object        
 8   session_duration_sec  6034 non-null   int64         
 9   device                6034 non-null   object        
 10  date                  6034 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(9)
memory usage: 518.7+ KB


,utm_campaign,device,session_duration_sec
0,Email_Welcome,tablet,522.339286
1,LinkedIn_Lead,tablet,499.382353
2,Organic_Direct,desktop,496.012048
3,Organic_Direct,tablet,493.975904
4,Organic_Direct,mobile,493.353659
5,FB_Awareness,tablet,490.028169
6,GSearch_Generic,mobile,489.043796
7,Organic_SEO,desktop,485.867470
8,GSearch_Generic,desktop,482.185393
9,TikTok_Promo,desktop,482.067164


#### Distribution of Ad Channels

#### Total Amount Spent per Channel

#### Distribution of Numerical Features (Amount Spent, Clicks, Impressions, CPC)

#### Ad Spend Over Time

In [ ]:
import pandas as pd
crm_df=pd.read_csv("crm_conversion_data.csv")
crm_df.head()
crm_df[(crm_df["country"]=="Australia")]["revenue_usd"].sum()
crm_df["customer_id"].nunique()
crm_df.columns
crm_df.groupby("country")["revenue_usd"].sum().sort_values(ascending=False)
crm_df.head()
crm_df["last_touch_campaign"].nunique()
crm_df[(crm_df["last_touch_campaign"]=="TikTok_Video1") & (crm_df["last_touch_channel"]=="Tiktok")]
##crm_df["last_touch_campaign"].nunique()
##1 Revenue Distribution by Last Touch Channel: Analyze which marketing channels are most effective in driving conversions and revenue.
crm_df.groupby("last_touch_channel")["revenue_usd"].sum().sort_values(ascending=False)
##Product Purchase Frequency: Determine the most frequently purchased products.
crm_df["product_purchased"].min()
##crm_df[(crm_df["product_purchased"]=="Pro Plan")]["conversion_date"].nunique()
##crm_df.groupby("product_purchased")["conversion_date"].nunique().sort_values(ascending=False)
crm_df[(crm_df["product_purchased"]=="Basic Plan")].nunique()
##Distribution of Revenue per Conversion
crm_df.head()
crm_df.groupby("product_purchased")["revenue_usd"].sum().sort_values(ascending=False)
rev =crm_df["revenue_usd"].sum()
prod =crm_df["product_purchased"].count()
b=rev/prod
crm_df["product_purchased"].isna().sum()
b
crm_df.head()
crm_df.groupby("last_touch_channel")["revenue_usd"].sum().sort_values(ascending=False)
crm_df["conversion_date"]=pd.to_datetime(crm_df["conversion_date"])
crm_df.info()
crm_df["conversion_month"]=crm_df["conversion_date"].dt.month
crm_df.groupby("conversion_month")["revenue_usd"].sum().sort_values(ascending=False)
crm_df["conversion_day"]=crm_df["conversion_date"].dt.day_name()
crm_df["conversion_day"]
crm_df.groupby(["conversion_day","product_purchased"])["revenue_usd"].sum().sort_values(ascending=False).reset_index()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          600 non-null    object        
 1   conversion_date      600 non-null    datetime64[ns]
 2   revenue_usd          600 non-null    float64       
 3   last_touch_channel   600 non-null    object        
 4   last_touch_campaign  572 non-null    object        
 5   product_purchased    600 non-null    object        
 6   country              600 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 32.9+ KB


,conversion_day,product_purchased,revenue_usd
0,Friday,Pro Plan,7569.87
1,Monday,Basic Plan,7431.77
2,Thursday,One-Time Purchase,7364.23
3,Tuesday,Pro Plan,6786.66
4,Saturday,One-Time Purchase,6703.06
5,Wednesday,Enterprise Plan,6526.81
6,Tuesday,Enterprise Plan,6510.54
7,Friday,Enterprise Plan,6293.91
8,Sunday,Basic Plan,6188.59
9,Saturday,Pro Plan,6178.89


In [ ]:
import pandas as pd
crm_df = pd.read_csv("crm_conversion_data.csv")
crm_df.head()
crm_df[(crm_df["country"]=="Australia")]["revenue_usd"].sum()
crm_df["customer_id"].nunique()
crm_df.columns
crm_df.groupby("country")["revenue_usd"].sum().sort_values(ascending=False)
crm_df.head()
##crm_df["last_touch_campaign"].nunique()
##1 Revenue Distribution by Last Touch Channel: Analyze which marketing channels are most effective in driving conversions and revenue.
crm_df.groupby("last_touch_channel")["revenue_usd"].sum().sort_values(ascending=False)
crm_df.tail(2)
crm_df["product_purchased"].value_counts()
crm_df["product_purchased"].nunique()
crm_df["revenue_usd"].sum()
crm_df["conversion_date"].nunique()
avg_rev_per_day=crm_df["revenue_usd"].sum()/crm_df["conversion_date"].nunique()


### Average Purchase Time (Purchases Per Day)

To calculate the average purchase time, or more precisely, the average number of purchases per day, we need to:
1.  Convert the `conversion_date` column to datetime objects.
2.  Determine the total number of unique days over which purchases occurred.
3.  Divide the total number of purchases by the number of days.

### Product Purchase Frequency

To find the most frequently purchased products, we can use `value_counts()` on the `product_purchased` column of the `crm_df`.

In [ ]:
product_frequency = crm_df['product_purchased'].value_counts()
product_frequency


,count
product_purchased,
One-Time Purchase,163
Basic Plan,150
Enterprise Plan,144
Pro Plan,143


### Calculating Session Duration

From the `web_log_df`, we can use the `session_duration_sec` column directly to understand the duration of each user session. If you are interested in the duration of the entire customer journey from the first touch to conversion, that would involve joining `web_log_df` and `crm_df` and calculating the time difference between the earliest `timestamp` for a `user_id` and their `conversion_date`.